<a href="https://colab.research.google.com/github/adalves-ufabc/2026.Q3-PLN/blob/main/2026_Q3_PLN_AULA_14_Notebook_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

## **Extração de Informação [LangChain]**
---



In [1]:
#@title Instalando o pacote LangChain
!pip install langchain

In [2]:
#@title Versão do LangChain

import langchain

print(langchain.__version__)

1.4.8


In [3]:
#@title Integração com o pacote da OpenAI

!pip install langchain-openai

In [4]:
#@title Definindo a chave da API da OpenAI
from getpass import getpass

OPENAI_API_KEY = getpass()

··········


## **Extração de Informação**

**Extração de Informação** (EI) é o processo de identificar e extrair dados estruturados a partir de texto não estruturado. O objetivo é transformar o texto bruto em informações organizadas que podem ser facilmente analisadas e usadas para outras tarefas, como a construção de bases de dados ou a realização de análises.

#### **Construir uma cadeia de extração**

***Schema***

Primeiro, precisamos descrever quais informações queremos extrair do texto.

Usaremos o `Pydantic` para definir um esquema de exemplo para extrair informações pessoais.

In [5]:
from typing import Optional

from pydantic import BaseModel, Field

class Pessoa(BaseModel):
    """Informações sobre uma pessoa."""

    # ^ Doc-string para a entidade Pessoa.
    # Este doc-string é enviado ao LLM como a descrição do esquema Pessoa,
    # e pode ajudar a melhorar os resultados da extração.

    # Observe que:
    # 1. Cada campo é `opcional` -- isso permite que o modelo opte por não extraí-lo!
    # 2. Cada campo tem uma `descrição` -- essa descrição é usada pelo LLM.
    # Ter uma boa descrição pode ajudar a melhorar os resultados da extração.
    nome: Optional[str] = Field(default=None, description="O nome da pessoa")
    cor_cabelo: Optional[str] = Field(
        default=None, description="A cor do cabelo da pessoa, se conhecida"
    )
    altura_metros: Optional[str] = Field(
        default=None, description="Altura medida em metros"
    )

Aqui estão duas boas práticas ao definir um esquema:

- Documente os atributos e o próprio esquema: essas informações são enviadas ao LLM e usadas para melhorar a qualidade da extração de informações.
- Não force o LLM a inventar informações! Acima, usamos `Optional` para os atributos, permitindo que o LLM retorne `None` se não souber a resposta.

**Extrator**

Vamos criar um extrator de informações usando o esquema definido acima.

In [6]:
from typing import Optional

from langchain_core.prompts import ChatPromptTemplate

# Defina um prompt personalizado para fornecer instruções e qualquer contexto adicional.
# 1) Você pode adicionar exemplos ao modelo de prompt para melhorar a qualidade da extração.
# 2) Introduza parâmetros adicionais para levar o contexto em consideração (por exemplo, incluir metadados
#    sobre o documento do qual o texto foi extraído.)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um algoritmo de extração especializado. "
            "Extraia apenas as informações relevantes do texto. "
            "Se você não souber o valor de um atributo solicitado para extração, "
            "retorne nulo para o valor do atributo.",
        ),
        ("human", "{texto}"),
    ]
)

Precisamos usar um modelo:

In [7]:
from langchain_openai import ChatOpenAI

modelo = ChatOpenAI(model="gpt-6-luna", api_key = OPENAI_API_KEY)

chain = prompt | modelo.with_structured_output( schema = Pessoa )

Vamos testar isso:

In [8]:
texto = "Alan Smith tem 183 centímetros de altura e cabelo loiro."

chain.invoke({"texto": texto})

Pessoa(nome='Alan Smith', cor_cabelo='loiro', altura_metros='1.83')

**Interessante**

>

Extração é Generativa 🤯
>
Os LLMs são modelos generativos, então eles podem fazer coisas bastante interessantes, como extrair corretamente a altura da pessoa em metros, mesmo que tenha sido fornecida em outra unidade!

**Múltiplas Entidades**

Na maioria dos casos, você deve estar extraindo uma lista de entidades em vez de uma única entidade. Isso pode ser facilmente alcançado usando o `Pydantic` ao aninhar modelos dentro de outros modelos.

In [9]:
from typing import Optional, List

from pydantic import BaseModel, Field

class Pessoa(BaseModel):
    """Informações sobre uma pessoa."""

    # ^ Doc-string para a entidade Pessoa.
    # Este doc-string é enviado ao LLM como a descrição do esquema Pessoa,
    # e pode ajudar a melhorar os resultados da extração.

    # Observe que:
    # 1. Cada campo é `opcional` -- isso permite que o modelo opte por não extraí-lo!
    # 2. Cada campo tem uma `descrição` -- essa descrição é usada pelo LLM.
    # Ter uma boa descrição pode ajudar a melhorar os resultados da extração.
    nome: Optional[str] = Field(default=None, description="O nome da pessoa")
    cor_cabelo: Optional[str] = Field(
        default=None, description="A cor do cabelo da pessoa, se conhecida"
    )
    altura_metros: Optional[str] = Field(
        default=None, description="Altura medida em metros"
    )

class Dados(BaseModel):
    """Dados extraídos sobre pessoas."""

    # Cria um modelo para que possamos extrair várias entidades.
    pessoas: List[Pessoa]

In [10]:
chain = prompt | modelo.with_structured_output( schema = Dados)

texto = "Meu nome é João, meu cabelo é preto e eu tenho 1,83 metros de altura. Ana tem a mesma cor de cabelo que eu."

resposta = chain.invoke({"texto": texto})

In [ ]:
resposta.pessoas

[Pessoa(nome='João', cor_cabelo='preto', altura_metros='1,83'),
 Pessoa(nome='Ana', cor_cabelo='preto', altura_metros=None)]

In [11]:
resposta.pessoas[0]

Pessoa(nome='João', cor_cabelo='preto', altura_metros='1,83')

In [12]:
resposta.pessoas[0].nome

'João'

**Página Web**

In [13]:
import requests

url = "https://www.worldometers.info/world-population/population-by-country/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.9"
}

pagina = requests.get(url, headers=headers, timeout=30)

print("Código HTTP:", pagina.status_code)
print("Tamanho do conteúdo:", len(pagina.text))
print("URL:", pagina.url)
print("\nConteúdo HTML:")
print(pagina.text[:100])

Código HTTP: 200
Tamanho do conteúdo: 438123
URL: https://www.worldometers.info/world-population/population-by-country/

Conteúdo HTML:
<!DOCTYPE html><html data-astro-transition-scope="astro-m2otfycz-2" lang="en" dir="ltr" class="highc


In [14]:
from pydantic import BaseModel, Field

# estrutura dos dados
class Pais(BaseModel):
    nome: str = Field(description="Nome do país")
    populacao: str = Field(description="População")
    area: str = Field(description="Área territorial em km²")
    densidade: str = Field(description="Densidade demográfica")

class Dados(BaseModel):
    paises: list[Pais]

In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

modelo = ChatOpenAI(model="gpt-6-luna", api_key = OPENAI_API_KEY)

prompt = ChatPromptTemplate.from_template("""
Analise o conteúdo HTML apresentado e extraia
as informações dos países.

Extraia os 10 primeiros países da tabela, identificando:
- Nome
- População
- Área territorial
- Densidade demográfica

Utilize exclusivamente as informações presentes no HTML.
Não invente ou complete informações ausentes.

HTML:
{texto}
""")

chain = prompt | modelo.with_structured_output(schema = Dados)

In [16]:
resposta = chain.invoke({"texto": pagina.text})

print(resposta.model_dump_json(indent=2))

{
  "paises": [
    {
      "nome": "India",
      "populacao": "1,476,625,576",
      "area": "2,973,193",
      "densidade": "497"
    },
    {
      "nome": "China",
      "populacao": "1,412,914,089",
      "area": "9,326,410",
      "densidade": "151"
    },
    {
      "nome": "United States",
      "populacao": "349,035,494",
      "area": "9,147,420",
      "densidade": "38"
    },
    {
      "nome": "Indonesia",
      "populacao": "287,886,782",
      "area": "1,811,569",
      "densidade": "159"
    },
    {
      "nome": "Pakistan",
      "populacao": "259,299,792",
      "area": "856,692",
      "densidade": "303"
    },
    {
      "nome": "Nigeria",
      "populacao": "242,431,832",
      "area": "910,768",
      "densidade": "266"
    },
    {
      "nome": "Brazil",
      "populacao": "213,562,666",
      "area": "8,358,140",
      "densidade": "26"
    },
    {
      "nome": "Bangladesh",
      "populacao": "177,818,044",
      "area": "130,170",
      "densidade": "1

#### **Como lidar com textos longos ao realizar a extração**

Quando trabalhar com arquivos, como PDFs, é provável que você encontre texto que excede a janela de contexto do seu modelo de linguagem. Para processar esse texto, considere estas estratégias:

1. **Mudar o LLM**: escolha um LLM diferente que suporte uma janela de contexto maior.
2. **Força Bruta**: divida o documento em partes e extraia o conteúdo de cada parte.
3. **RAG (*Retrieval-Augmented Generation*)**: divida o documento em partes, indexe as partes e extraia conteúdo apenas de um subconjunto de partes que pareçam "relevantes".

Lembre-se de que essas estratégias têm diferentes compromissos, e a melhor estratégia provavelmente depende da aplicação que você está projetando!

Aqui iremos demonstrar como implementar as estratégias 2 e 3.

Precisamos de alguns dados de exemplo! Vamos baixar um artigo sobre carros da **Wikipedia** e carregá-lo como um `Document` do LangChain.

In [11]:
import re
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/115.0.0.0 Safari/537.36'
}

url = 'https://en.wikipedia.org/wiki/History_of_the_automobile'

pagina = requests.get(url, headers=HEADERS, timeout=30)

# extrair o texto do HTML
soup = BeautifulSoup(pagina.text, "html.parser")

texto = soup.get_text(separator="\n")

# remover linhas em branco consecutivas
texto = re.sub(r"\n\n+", "\n", texto)

# criar documento LangChain
documento = Document(page_content=texto)

print(documento.page_content[:100])


History of the automobile - Wikipedia
Jump to content
Main menu
Main menu
move to sidebar
hide
		Na


In [18]:
documento.page_content[:1000]

'\nHistory of the automobile - Wikipedia\nJump to content\nMain menu\nMain menu\nmove to sidebar\nhide\n\t\tNavigation\n\t\nMain page\nContents\nCurrent events\nRandom article\nAbout Wikipedia\nContact us\n\t\tContribute\n\t\nHelp\nLearn to edit\nCommunity portal\nRecent changes\nUpload file\nSpecial pages\nSearch\nSearch\nAppearance\nDonate\nCreate account\nLog in\nPersonal tools\nDonate\nCreate account\nLog in\nContents\nmove to sidebar\nhide\n(Top)\n1\nPower sources\nToggle Power sources subsection\n1.1\nSteam-powered wheeled vehicles\n1.1.1\n17th and 18th centuries\n1.1.2\n19th century\n1.1.3\n20th century\n1.2\nElectric automobiles\n1.2.1\n19th century\n1.2.2\n20th century\n1.2.3\n21st century\n1.3\nInternal combustion engines\n1.3.1\nGas mixtures\n1.3.2\nGasoline\n2\nEras of invention\nToggle Eras of invention subsection\n2.1\nHorseless carriage or veteran era\n2.2\nBrass/Edwardian era\n2.3\nVintage era\n2.4\nPre-war era\n2.5\nPostwar era\n2.6\nModern era\n3\nSee also\n4\nReferen

In [19]:
print(len(documento.page_content))

74979


**Definir o esquema**

Usaremos o `Pydantic` para definir o esquema das informações que desejamos extrair. Neste caso, extrair uma lista de principais desenvolvimentos (por exemplo, eventos históricos importantes) que incluam um ano e uma descrição.

>
Observe que também incluímos uma chave de evidência e instruímos o modelo a fornecer, literalmente, as sentenças relevantes do texto do artigo. Isso nos permite comparar os resultados da extração com (a reconstrução do modelo de) texto do documento original.

In [16]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Desenvolvimento(BaseModel):
    """Informações sobre um desenvolvimento na história dos carros."""

    ano: int = Field(
        ..., description="O ano em que houve um desenvolvimento histórico importante."
    )
    descricao: str = Field(
        ..., description="O que aconteceu neste ano? Qual foi o desenvolvimento?"
    )
    evidencia: str = Field(
        ...,
        description="Repita literalmente a(s) frase(s) da qual as informações de ano e descrição foram extraídas.",
    )


class ExtracaoDados(BaseModel):
    """Informações extraídas sobre desenvolvimentos chave na história dos carros."""

    desenvolvimentos: List[Desenvolvimento]


# Defina um prompt personalizado para fornecer instruções e qualquer contexto adicional.
# 1) Você pode adicionar exemplos no template do prompt para melhorar a qualidade da extração.
# 2) Introduza parâmetros adicionais para considerar o contexto (por exemplo, incluir metadados
#    sobre o documento do qual o texto foi extraído.)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um especialista em identificar desenvolvimentos históricos chave em textos. "
            "Extraia apenas desenvolvimentos históricos importantes. Não extraia nada se não houver informações importantes no texto.",
        ),
        ("human", "{texto}"),
    ]
)

**Criar um extrator**

Vamos selecionar um LLM. Como estamos usando chamadas de `ferramentas`, precisaremos de um modelo que suporte esse recurso.

In [17]:
from langchain_openai import ChatOpenAI

modelo = ChatOpenAI(model="gpt-6-luna", api_key=OPENAI_API_KEY)

In [18]:
extrator = prompt | modelo.with_structured_output(
    schema = ExtracaoDados,
    include_raw = False,
)

**Abordagem de força bruta**



Divida os documentos em partes (*chunks*) de modo que cada parte se encaixe na janela de contexto dos LLMs.

In [8]:
!pip install langchain-text-splitters

In [25]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    chunk_size=2000,
    chunk_overlap=20,
)

textos = text_splitter.split_text(documento.page_content)

* `chunk_size=2000`: define o tamanho máximo de cada parte (*chunk*) em tokens. Neste caso, cada parte terá no máximo 2000 tokens.

* `chunk_overlap=20`: define o número de tokens que se sobrepõem entre partes adjacentes. Uma sobreposição de 20 tokens significa que os 20 tokens finais de uma parte serão incluídos no início da próxima parte. Isso ajuda a garantir que o contexto não seja perdido entre partes consecutivas.

Use a funcionalidade de processamento em lote (`batch`) para executar a extração em paralelo em cada parte!

In [26]:
# Limite apenas às primeiras 3 partes
# para que o código possa ser executado rapidamente
partes = textos[:3]

resposta = extrator.batch(
    [{"texto": texto} for texto in partes],
    {"max_concurrency": 5},  # número máximo de chamadas simultâneas
)

**Mesclar resultados**

Após extrair dados das partes, será necessário mesclar as extrações.

In [27]:
desenvolvimentos = []

for extracao in resposta:
    desenvolvimentos.extend(extracao.desenvolvimentos)

desenvolvimentos[:10]

[Desenvolvimento(ano=1649, descricao='Hans Hautsch construiu uma carruagem movida por mecanismo de relojoaria.', evidencia='In 1649, Hans Hautsch of Nuremberg built a clockwork-driven carriage.'),
 Desenvolvimento(ano=1672, descricao='Ferdinand Verbiest criou um veículo a vapor em pequena escala, considerado possivelmente o primeiro veículo a vapor funcional.', evidencia='In 1672, a small-scale steam-powered vehicle was created by Ferdinand Verbiest;'),
 Desenvolvimento(ano=1769, descricao='Nicolas-Joseph Cugnot construiu o primeiro automóvel a vapor capaz de transportar pessoas.', evidencia='the first steam-powered automobile capable of human transportation was built by Nicolas-Joseph Cugnot in 1769.'),
 Desenvolvimento(ano=1826, descricao='Samuel Brown testou o primeiro motor de combustão interna aplicado industrialmente.', evidencia='Samuel Brown later tested the first industrially applied internal combustion engine in 1826. Only two of these were made.'),
 Desenvolvimento(ano=1886,

In [28]:
for desenvolvimento in sorted(desenvolvimentos, key=lambda x: x.ano):
    print(f"{desenvolvimento.ano}: {desenvolvimento.descricao}")

1649: Hans Hautsch construiu uma carruagem movida por mecanismo de relojoaria.
1672: Ferdinand Verbiest criou um veículo a vapor em pequena escala, considerado possivelmente o primeiro veículo a vapor funcional.
1769: Nicolas-Joseph Cugnot construiu o primeiro automóvel a vapor capaz de transportar pessoas.
1806: François Isaac de Rivaz construiu um motor movido pela combustão interna de uma mistura de hidrogênio e oxigênio.
1816: Josef Bozek construiu um automóvel a vapor movido a óleo.
1826: Samuel Brown testou o primeiro motor de combustão interna aplicado industrialmente.
1826: Samuel Brown testou seu motor de combustão interna alimentado a hidrogênio para impulsionar um veículo em Shooter's Hill, em Londres.
1828: Ányos Jedlik construiu um pequeno modelo de carro movido por seu motor elétrico recém-inventado.
1834: Thomas Davenport instalou seu motor elétrico de corrente contínua em um pequeno modelo de carro, que funcionava numa pista circular eletrificada.
1835: Sibrandus Strati

**Abordagem baseada em *RAG***

***RAG*** (*Retrieval-Augmented Generation* - Geração aumentada de recuperação)

Outra ideia simples é dividir o texto em partes, mas em vez de extrair informações de cada parte, concentre-se apenas nas partes mais relevantes.


**Atenção**

Pode ser difícil identificar quais partes são relevantes.

Por exemplo, no artigo sobre carros que estamos usando aqui, a maior parte do artigo contém informações sobre desenvolvimentos chave. Então, ao usar ***RAG***, provavelmente estaremos descartando muitas informações relevantes.

Sugerimos experimentar com seu caso de uso e determinar se essa abordagem funciona ou não.

Para implementar a abordagem baseada em ***RAG***:

1. Divida seu(s) documento(s) em partes e indexe-os (por exemplo, em um `vectorstore`);
2. Adicione um passo de recuperação antes do extrator, utilizando o `vectorstore`.

***RAG*** (*Retrieval-Augmented Generation*) é uma abordagem que combina recuperação de informações com geração de texto. A ideia principal é melhorar a capacidade de um modelo de linguagem ao integrar informações recuperadas de uma base de dados ou índice com a geração de texto.

Como funciona o ***RAG***:

   * **Recuperação**: quando um modelo precisa gerar uma resposta ou extrair informações, primeiro ele realiza uma busca em uma base de dados ou índice para recuperar partes relevantes do texto ou documentos que podem ajudar a responder a pergunta ou completar a tarefa. Esta base de dados é frequentemente um `vectorstore`, que armazena *embeddings* de texto para rápida recuperação.

   * **Geração**: após recuperar as informações relevantes, o modelo de linguagem utiliza essas informações como contexto adicional para gerar uma resposta mais informada e precisa. Isso combina o conhecimento do modelo com dados específicos recuperados, aumentando a relevância e precisão da saída gerada.

***FAISS*** (*Facebook AI Similarity Search*) é uma biblioteca desenvolvida pelo Facebook AI Research para realizar busca eficiente e similaridade de alta dimensão em grandes conjuntos de dados. É projetada para lidar com grandes volumes de dados e facilitar a recuperação de informações baseadas em similaridade.

***FAISS*** é especializado em encontrar vetores (representações numéricas de dados) similares a um vetor de consulta.

***Vectorstore*** é um termo usado para descrever um tipo de armazenamento especializado em vetores, frequentemente utilizado em sistemas de recuperação de informações e modelos de aprendizado de máquina. Ele é projetado para gerenciar e buscar vetores de alta dimensão de maneira eficiente.

In [5]:
!pip install faiss-cpu

In [6]:
!pip install langchain-community

Aqui está um exemplo simples que utiliza o `vectorstore` `FAISS`.

In [12]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter, TokenTextSplitter

# Instancia o text_splitter para evitar o NameError
text_splitter = TokenTextSplitter(
    chunk_size=2000,
    chunk_overlap=20,
)

textos = text_splitter.split_text(documento.page_content)
vectorstore = FAISS.from_texts(textos, embedding = OpenAIEmbeddings(api_key=OPENAI_API_KEY))

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 1}
)  # extrai apenas do primeiro documento

Neste caso, o extrator ***RAG*** está analisando apenas o principal documento.

In [19]:
rag_extrator = {
    "texto": retriever | (lambda docs: docs[0].page_content)
} | extrator

In [20]:
resposta = rag_extrator.invoke("Principais desenvolvimentos associados aos carros")

In [21]:
for desenvolvimento in resposta.desenvolvimentos:
    print(desenvolvimento)

ano=1959 descricao='O Mini apareceu com um projeto inovador que combinava motor transversal, tração dianteira, suspensão independente e carroceria monobloco.' evidencia="BMC's space-saving and trend-setting transverse engine, front-wheel-drive, independent suspension and monocoque bodied Mini, which first appeared in 1959, was marketed under the Austin and Morris names, until Mini became a marque in its own right in 1969."
ano=1969 descricao='O Mini passou a ser uma marca própria, após ser comercializado sob os nomes Austin e Morris.' evidencia="BMC's space-saving and trend-setting transverse engine, front-wheel-drive, independent suspension and monocoque bodied Mini, which first appeared in 1959, was marketed under the Austin and Morris names, until Mini became a marque in its own right in 1969."
ano=1973 descricao='A crise do petróleo, junto a novas exigências de emissões e segurança e à expansão das exportações japonesas e europeias, contribuiu para transformar o setor automotivo.' 

In [22]:
for desenvolvimento in sorted(resposta.desenvolvimentos, key=lambda x: x.ano):
    print(f"{desenvolvimento.ano}: {desenvolvimento.descricao}")

1959: O Mini apareceu com um projeto inovador que combinava motor transversal, tração dianteira, suspensão independente e carroceria monobloco.
1966: O Oldsmobile Toronado foi o primeiro carro americano da era moderna com tração dianteira e também introduziu freios ABS eletrônicos e airbag.
1969: O Mini passou a ser uma marca própria, após ser comercializado sob os nomes Austin e Morris.
1973: A crise do petróleo, junto a novas exigências de emissões e segurança e à expansão das exportações japonesas e europeias, contribuiu para transformar o setor automotivo.
1983: As minivans da Chrysler popularizaram o formato de carroceria de dois volumes e quase substituíram as peruas no mercado.
1984: O Renault Espace foi o primeiro carro de volume único produzido em massa na categoria de MPV não comercial.
1990: A partir dos anos 1990, a popularidade mundial de SUVs e crossovers alterou o mercado e contribuiu para a queda nas vendas de outros tipos de carroceria.


**IMPORTANTE:**

Métodos diferentes têm suas próprias vantagens e desvantagens relacionadas a custo, desempenho e precisão.

Fique atento a esses problemas:

- Dividir o conteúdo em partes significa que o LLM pode falhar em extrair informações se essas informações estiverem espalhadas por vários fragmentos.
- Um grande sobreposição entre os fragmentos pode fazer com que a mesma informação seja extraída duas vezes, então esteja preparado para desduplicar!
- LLMs podem inventar dados. Se você estiver procurando por um único fato em um texto longo e usar uma abordagem de força bruta, pode acabar obtendo mais dados inventados.

#### **Como usar apenas o *prompting* para realizar a extração**

Recursos de chamada de `ferramentas` não são necessários para gerar saídas estruturadas a partir de LLMs. LLMs que conseguem seguir bem as instruções do *prompt* podem ser encarregados de fornecer informações em um formato específico.
>
Essa abordagem depende de projetar bons *prompts* e, em seguida, fazer o *parser* da saída dos LLMs para garantir que eles extraíam informações de forma eficaz.
>
Para extrair dados sem recursos de chamada de `ferramentas`:
>
1. Instrua o LLM a gerar texto seguindo um formato esperado (por exemplo, `JSON` com um determinado esquema);
2. Use *parsers* de saída para estruturar a resposta do modelo em um objeto Python desejado.

Nem todos os modelos suportam `.with_structured_output()`, já que nem todos os modelos têm suporte para chamadas de `ferramentas` ou modo `JSON`. Para esses modelos, você precisará solicitar diretamente ao modelo que use um formato específico e usar um *parser* de saída para extrair a resposta estruturada da saída bruta do modelo.


**Usando `PydanticOutputParser`**

O exemplo a seguir usa o `PydanticOutputParser` embutido para fazer o *parser* da saída de um modelo de *chat* solicitado a corresponder ao esquema `Pydantic` fornecido. Note que estamos adicionando `format_instructions` diretamente ao *prompt* a partir de um método no *parser*:

In [23]:
from typing import List

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class Pessoa(BaseModel):
    """Informação sobre uma pessoa."""

    nome: str = Field(..., description="O nome da pessoa.")
    altura_metros: float = Field(
        ..., description="A altura da pessoa expressa em metros."
    )

class Pessoas(BaseModel):
    """Informações de identificação sobre todas as pessoas em um texto."""

    pessoas: List[Pessoa]

# parser
parser = PydanticOutputParser(pydantic_object=Pessoas)

# prompt
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Responda à consulta do usuário. Envolva a saída em tags `json`\n{format_instructions}",
        ),
        ("human", "{consulta}"),
    ]
).partial(format_instructions = parser.get_format_instructions())

Vamos dar uma olhada em quais informações serão enviadas ao modelo:

In [24]:
consulta = "Ana tem 23 anos e mede 1,83 metros de altura."

print(prompt.invoke(consulta).to_string())

System: Responda à consulta do usuário. Envolva a saída em tags `json`
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"Pessoa": {"description": "Informação sobre uma pessoa.", "properties": {"nome": {"description": "O nome da pessoa.", "title": "Nome", "type": "string"}, "altura_metros": {"description": "A altura da pessoa expressa em metros.", "title": "Altura Metros", "type": "number"}}, "required": ["nome", "altura_metros"], "title": "Pessoa", "type": "object"}}, "description": "Informações de identificação sobre todas as pessoas em um texto.", "properties": {"pessoas": {"ite

E agora, vamos invocá-lo:

In [25]:
# Ajustando o prompt para não forçar as tags <json> </json> customizadas que quebram o parser padrão
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Responda à consulta do usuário estritamente no formato JSON instruído abaixo:\n{format_instructions}",
        ),
        ("human", "{consulta}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | modelo | parser

resposta = chain.invoke({"consulta": consulta})

In [26]:
resposta

Pessoas(pessoas=[Pessoa(nome='Ana', altura_metros=1.83)])

In [27]:
type(resposta)

__main__.Pessoas

In [28]:
resposta.pessoas[0].nome

'Ana'

In [29]:
for pessoa in resposta.pessoas:
    print(pessoa.nome)

Ana


***Parser* Personalizado**

Você também pode criar um *prompt* e um *parser* personalizados com a **Linguagem de Expressão LangChain** (`LCEL`), usando uma função simples para fazer o *parser* da saída do modelo:

In [30]:
import json
import re
from typing import List

from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class Pessoa(BaseModel):
    nome: str = Field(description="O nome da pessoa.")
    altura_metros: float = Field(
        description="A altura da pessoa expressa em metros."
    )

class Pessoas(BaseModel):
    pessoas: List[Pessoa]

# Prompt
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Responda à consulta do usuário utilizando exclusivamente
        as informações fornecidas.

        Retorne a resposta em formato JSON, seguindo o esquema:
        {schema}

        Não inclua explicações adicionais."""
    ),
    ("human", "{consulta}")
]).partial(schema=json.dumps(Pessoas.model_json_schema(), ensure_ascii=False))

# Parser personalizado
def extrair_json(mensagem: AIMessage) -> dict:

    texto = mensagem.content

    # Procurar JSON com ou sem bloco Markdown
    padrao = r"```json\s*(.*?)\s*```"
    matches = re.findall(padrao, texto, re.DOTALL)

    if matches:
        texto = matches[0]
    else:
        texto = texto.strip()

    try:
        return json.loads(texto)
    except Exception:
        raise ValueError(f"Falha no parser: {mensagem}")

In [31]:
texto = "Ana tem 23 anos e mede 1,83 metros de altura."

print(prompt.format_prompt(consulta = texto).to_string())

System: Responda à consulta do usuário utilizando exclusivamente
        as informações fornecidas.

        Retorne a resposta em formato JSON, seguindo o esquema:
        {"$defs": {"Pessoa": {"properties": {"nome": {"description": "O nome da pessoa.", "title": "Nome", "type": "string"}, "altura_metros": {"description": "A altura da pessoa expressa em metros.", "title": "Altura Metros", "type": "number"}}, "required": ["nome", "altura_metros"], "title": "Pessoa", "type": "object"}}, "properties": {"pessoas": {"items": {"$ref": "#/$defs/Pessoa"}, "title": "Pessoas", "type": "array"}}, "required": ["pessoas"], "title": "Pessoas", "type": "object"}

        Não inclua explicações adicionais.
Human: Ana tem 23 anos e mede 1,83 metros de altura.


In [32]:
chain = prompt | modelo | extrair_json

resposta = chain.invoke({"consulta": texto})

In [33]:
resposta

{'pessoas': [{'nome': 'Ana', 'altura_metros': 1.83}]}

In [34]:
type(resposta)

dict

In [35]:
print(resposta["pessoas"][0]["nome"])
print(resposta["pessoas"][0]["altura_metros"])

Ana
1.83
